# Amzon Bedrock でモデルを呼び出す各パターンを試す

In [ ]:
import boto3
import json

bedrock_runtime = boto3.client(service_name='bedrock-runtime', region_name='ap-northeast-1')

## Invoke

Amazon Nova Lite2 を呼び出す

In [ ]:
modelId = "jp.amazon.nova-2-lite-v1:0"

response = bedrock_runtime.invoke_model(
    modelId=modelId,
    body=json.dumps({
        "messages": [{
            "role": "user",
            "content": [
                {"text": "こんにちは"}
            ]
        }],
        "inferenceConfig": {
            "maxTokens": 2048,
            "temperature": 0
        }
    })
)

response = json.loads(response.get('body').read())
print(response["output"]["message"]["content"][0]["text"])

Anthropic Claude Haiku 4.5 を呼び出す。

In [ ]:
modelId = "jp.anthropic.claude-haiku-4-5-20251001-v1:0"

response = bedrock_runtime.invoke_model(
    modelId=modelId,
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "messages": [{
            "role": "user",
            "content": [
                { "type": "text", "text": "こんにちは" }
            ]
        }],
        "max_tokens": 2048,
        "temperature": 0
    })
)

response = json.loads(response.get('body').read())
print(response["content"][0]["text"])

このように、invoke_model がモデルごとに異なるフォーマットの body を指定する必要があったり、レスポンスのフォーマットも様々です。

そのため、invoke_model ではなく、共通の入出力フォーマットで各モデルを扱うことのできる converse の利用が推奨されています。

https://docs.aws.amazon.com/ja_jp/bedrock/latest/userguide/conversation-inference.html
> メッセージをサポートするすべての Amazon Bedrock モデルで動作する一貫した API を提供する Converse API を使用することをお勧めします。そうすることで、コードを 1 回だけ記述し、それをさまざまなモデルで使用できます。

## Converse

共通の入出力フォーマット（[参考](https://docs.aws.amazon.com/ja_jp/bedrock/latest/userguide/conversation-inference-call.html)）で各モデルを扱うことができます。
しかし、いくつかのモデルは Converse に対応していません（[参考](https://docs.aws.amazon.com/ja_jp/bedrock/latest/userguide/conversation-inference-supported-models-features.html)）

In [ ]:
modelId = "jp.amazon.nova-2-lite-v1:0"
# modelId = "jp.anthropic.claude-haiku-4-5-20251001-v1:0"

response = bedrock_runtime.converse(
    modelId = modelId,
    messages = [{
        "role": "user",
        "content": [{"text": "こんにちは"}],
    }],
    inferenceConfig = {
        "maxTokens": 2048,
        "temperature": 0
    }
)

print(response["output"]["message"]["content"][0]["text"])

## ConverseStream

出力を少しずつ取得して反映する、ストリーム形式でモデルを呼び出すことができます。

このとき、入力フォーマットは Converse と変わらず、出力のみが変化します。

In [ ]:
modelId = "jp.amazon.nova-2-lite-v1:0"
# modelId = "jp.anthropic.claude-haiku-4-5-20251001-v1:0"

response = bedrock_runtime.converse_stream(
    modelId = modelId,
    messages = [{
        "role": "user",
        "content": [{"text": "AWS について紹介して"}],
    }],
    inferenceConfig = {
        "maxTokens": 2048,
        "temperature": 0
    }
)

for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"], end="")